# LangChain Lesson Generator

An idiomatic **LangChain** rewrite of the chemistry lesson generator.

Instead of calling the model directly with `llm.invoke(...)`, this version uses the core LangChain building blocks:

- `ChatPromptTemplate` — declarative system + human prompt with named variables
- LangChain Expression Language (LCEL) — `prompt | llm | parser` pipeline
- `StrOutputParser` — turns the chat message into a plain string

The model runs fully locally through **Ollama** (`qwen2.5:14b`), so no API key is required.

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from docx import Document
import re
import os

/Users/shubhamsoni/shubham/AlphaChem/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
#  Pick the model provider:
#   'azure'  -> Azure OpenAI  (needs AZURE_OPENAI_* vars in .env)
#   'openai' -> standard OpenAI (needs OPENAI_API_KEY)
#   'ollama' -> local model    (no key)
PROVIDER = 'azure'

if PROVIDER == 'azure':
    # Read Azure settings from .env. .strip() guards against stray spaces after '='.
    azure_endpoint = os.environ['AZURE_OPENAI_ENDPOINT'].strip()
    azure_deployment = os.environ['AZURE_OPENAI_DEPLOYMENT'].strip()
    azure_api_key = os.environ['AZURE_OPENAI_API_KEY'].strip()
    api_version = os.environ.get('OPENAI_API_VERSION', '2024-10-21').strip()

    # Load the LangChain Azure chat model
    llm = AzureChatOpenAI(
        azure_endpoint=azure_endpoint,
        azure_deployment=azure_deployment,
        api_key=azure_api_key,
        api_version=api_version,
        temperature=0.7,
        max_tokens=8192,
    )
    print(f'Using Azure OpenAI deployment: {azure_deployment}')
elif PROVIDER == 'openai':
    if not os.environ.get('OPENAI_API_KEY'):
        raise RuntimeError('OPENAI_API_KEY is not set. Add it to your .env file.')
    llm = ChatOpenAI(
        model='gpt-4o-mini',
        temperature=0.7,
        max_tokens=8192,
    )
    print('Using OpenAI model via LangChain: gpt-4o-mini')
else:
    llm = ChatOllama(
        model='qwen2.5:14b',
        temperature=0.7,
        num_predict=8192,
    )
    print('Using local Ollama model via LangChain: qwen2.5:14')

NameError: name 'AzureChatOpenAI' is not defined

In [ ]:
# Local chat model served by Ollama — no API key needed
llm = ChatOllama(
    model='qwen2.5:14b',
    temperature=0.7,
    num_predict=8192,  # max tokens to generate
)
print('Using local Ollama model via LangChain: qwen2.5:14b')

## Lesson inputs

Plain-text values that describe the lesson. These are passed into the prompt template as named variables.

In [ ]:
reading_score = '''Please follow these formula in order to generate content. Flesch Reading Ease Score >= 80, and grade level 8
1. Flesch Reading Ease Score = 206.835 − 1.015 × ( Total Words / Total Sentences ) − 84.6 × ( Total Syllables / Total Words )
2. Flesch-Kincaid Grade Level = 0.39 × ( Total Words / Total Sentences ) + 11.8 × ( Total Syllables / Total Words ) − 15.59
'''

lesson_objective = '''Describe how ions are formed.
 Write the symbols and charges of ions and octet rule and its exceptions
 Predict the charge of an ion based on its position on the periodic table.'''

lesson_vocabulary = '''Octet Rule
 Anion
 Cation
 Electrolyte
 Electron affinity
 Ionic radius
 Ionization
 Octet rule'''

essential_question = '''How are ions formed, and what role do they play in chemical bonding?'''

performance_expectations = '''HS-PS1-2: Construct and revise an explanation for the outcome of a simple chemical reaction based on the outermost electron states of atoms, trends in the periodic table, and knowledge of the patterns of chemical properties.'''

disciplinary_core_ideas = '''PS1.A: Structure and Properties of Matter. The structure and interactions of matter at the bulk scale are determined by electrical forces within and between atoms.'''

phenomenon = '''Unit phenomenon: Danger! Icy Roads
In northern countries, where winter brings extremely cold weather, streets and roads are often covered in ice and snow. This creates hazardous conditions for both pedestrians and drivers. Pedestrians can slip and fall, risking injury, while cars may skid on the icy surfaces, potentially causing accidents. To reduce these dangers, road salt is spread on icy streets to help melt the ice and snow. As the salt comes into contact with the ice, ice and snow seem to vanish. Metal street signs and lampposts are also exposed to the same ice and snow, but they do not vanish.
Chapter Phenomenon: Salt vs. Metal, Why Does Water Treat Them Differently?
When road salt is spread on icy and snowy streets, the ice and snow melt and the salt dissolves in the water. Street signs and lampposts are made of metal, but they do not melt the snow or dissolve in water. Instead, they remain intact, showing no immediate signs of rust or corrosion. Why do salt and metal behave so differently with water?
'''

## Prompt template

A `ChatPromptTemplate` holds a system message (writing persona + readability target) and a human message (the structured lesson request). Variables in `{curly_braces}` are filled in at invocation time.

In [ ]:
system_template = (
    "You are a chemistry textbook writer for ages 14-15. The content should be easy to read, "
    "so please use these formulas to keep the language simple, aiming for a Flesch Reading Ease "
    "Score greater than 90:\n{reading_score}\n"
    "Write in a way that feels human-authored. The target audience is USA school grade 9."
)

human_template = """Generate a detailed and structured lesson plan for "{lesson_name}" in Chapter "{chapter_name}" of Unit "{unit_name}".
The content should be structured, consistent, and align with the following points:
 - lesson objective: {lesson_objective}
 - lesson vocabulary: {lesson_vocabulary}
 - Essential Question: {essential_question}
 - Performance Expectations: {performance_expectations}
 - Disciplinary Core Ideas: {disciplinary_core_ideas}

## Unit Title
## Chapter Title
# Lesson Title

### 1. Big Idea
- One line that addresses the main concept of the lesson.
- A subordinate of the Chapter's Big Idea that addresses the main concepts in the lesson.

### 2. Essential Questions
- Include the following Essential Question(s) to encourage critical thinking:
    - {essential_question}

### 3. Phenomenon-Based Learning
- The lesson should build upon the chapter's storyline and introduce a specific aspect, question, or issue explored through hands-on tasks.
- Phenomenon: {phenomenon}

### 4. Vocabulary
- Define these key terms to support students' understanding:
    - {lesson_vocabulary}

### 5. SMART Objectives
- List 3-4 Specific, Measurable, Achievable, Relevant, Time-based objectives from the lesson objective:
    - {lesson_objective}

### 6. Engage (Ignite)
- Start with a phenomenon-related question or task to grab attention and continue the same story.
- Include one hands-on experiment relevant to the lesson topic, with a step-by-step procedure.
- Add 2-3 follow-up questions based on the activity.

### 7. Pre-Explore (Direct Instruction)
- Provide background information linking the phenomenon and key concepts.
- Use interactive elements (notes, discussions, scaffolded questions) to break up the content.

### 8. Evaluate (Progress Check) - Pre-Explore
- Frame up to 3 scaffolded questions (DOK 1-3) to connect concepts to the hands-on activity.

### 9. Explore (Pathfinder)
- Guide students through a hands-on activity with clear instructions.
- Ensure they collect data and engage in group discussions.
- Use retrieval practice (quizzes or questions) to reinforce learning.

### 10. Explain (Lightbulb)
- This section should be around 4500 words and explain the core concept of the lesson based on the storyline.
- Ensure content follows the unit and chapter storyline, aligning with the lesson objective and the Big Idea.
- Break down complex concepts into structured sections, making them easy to digest for 14-15 year olds.
- For every main concept explained, include one sample solved problem where applicable, then one question for students to solve as a Progress Check.

### 11. Evaluate (Progress Check) - Explain
- Include 3 scaffolded questions (DOK 1-3) to confirm understanding of key concepts covered in the Explain section.

### 12. Elaborate (Power Up)
- Pose mini-tasks or open-ended questions encouraging deeper thinking.

### 13. Final Evaluation
- Provide 1 debate question, including arguments and points for discussion.
- Frame 8 assessment questions:
    - 4 multiple-choice questions (with options and correct answers).
    - 4 long-answer questions requiring application of knowledge.

### 14. Extend (Beyond the Lesson) [Optional]
- Suggest additional tasks, readings, or challenges related to the lesson.
- Provide opportunities for spaced practice to reinforce previously learned key concepts.
"""

prompt = ChatPromptTemplate.from_messages([
    ('system', system_template),
    ('human', human_template),
])

## Build the chain (LCEL)

The pipe operator wires the components into a runnable chain: the prompt is rendered, sent to the model, and the response is parsed into a string.

In [ ]:
chain = prompt | llm | StrOutputParser()

In [ ]:
# Pull the first integer out of a name, e.g. 'Unit 2: ...' -> '2'
def extract_number(text):
    return re.search(r'\d+', text).group()

# Save generated content to a .docx named like U2Ch6L1.docx
def save_lesson_content_to_docx(unit_name, chapter_name, lesson_name, lesson_content):
    doc = Document()
    doc.add_heading(f'Unit: {unit_name}', level=1)
    doc.add_heading(f'Chapter: {chapter_name}', level=2)
    doc.add_heading(f'Lesson: {lesson_name}', level=3)
    doc.add_paragraph(lesson_content)

    file_name = f'U{extract_number(unit_name)}Ch{extract_number(chapter_name)}L{extract_number(lesson_name)}.docx'
    doc.save(file_name)
    print(f'Content saved to {file_name}')
    return file_name

## Generate and save a lesson

In [ ]:
unit_name = 'Unit 2: Atomic Structure and Bonding'
chapter_name = 'Chapter 6: Ionic and Metallic Bonding'
lesson_name = 'Lesson 1: Formation and Properties of Ions'

# Invoke the chain — pass every template variable as a dict
lesson_content = chain.invoke({
    'unit_name': unit_name,
    'chapter_name': chapter_name,
    'lesson_name': lesson_name,
    'reading_score': reading_score,
    'lesson_objective': lesson_objective,
    'lesson_vocabulary': lesson_vocabulary,
    'essential_question': essential_question,
    'performance_expectations': performance_expectations,
    'disciplinary_core_ideas': disciplinary_core_ideas,
    'phenomenon': phenomenon,
})

print(lesson_content)
save_lesson_content_to_docx(unit_name, chapter_name, lesson_name, lesson_content)